# Task 1: PID Stabilization of an Inverted Pendulum on a Cart

---

The inverted pendulum is unstable by nature. A small perturbation from the upright position causes the pendulum to fall — unless an active controller applies force to the cart in real time. This notebook derives the mathematical model from first principles, builds a PID controller, and demonstrates stabilization through simulation.

The physical setup: a cart of mass $m_c$ moves along a horizontal rail. A rigid rod of mass $m_p$ and length $l_s$ (to center of mass) is attached to the cart via a pivot. The only control input is a horizontal force $F$ applied to the cart. Friction acts on both the cart ($k_c$) and the pendulum joint ($k_p$).

## 1. Dependencies and Setup

In [ ]:
# --- Setup Paths --- #
import sys
sys.path.append("../")
sys.path.append("../model")

# --- General Imports --- #
import numpy as np
import matplotlib.pyplot as plt

# --- Internal Modules --- #
from model.pendulum import InvertedPendulum
from aux.animate import showAnimation

## 2. The Physical Model

### 2.1 State Variables

The system has 4 degrees of freedom packed into a single state vector:

$$\boldsymbol{x} = \begin{bmatrix} x_c \\ \dot{x}_c \\ \theta \\ \dot{\theta} \end{bmatrix}$$

where:
- $x_c$ — horizontal position of the cart [m]
- $\dot{x}_c$ — velocity of the cart [m/s]
- $\theta$ — angle of the pendulum from the vertical [rad] ($\theta = 0$ means upright)
- $\dot{\theta}$ — angular velocity of the pendulum [rad/s]

Only $x_c$ and $\theta$ are directly measurable. The velocities $\dot{x}_c$ and $\dot{\theta}$ remain hidden — a constraint that matters for controller design.

### 2.2 Sign Convention

The angle $\theta$ is measured from the upright vertical. When $\theta = 0$, the pendulum points straight up. Positive $\theta$ corresponds to a **counterclockwise** rotation from vertical — the pendulum leans in the **negative** $x$-direction (to the left). The force $F > 0$ pushes the cart to the right.

### 2.3 Lagrangian Derivation

The equations of motion come from the Euler-Lagrange formalism. The generalized coordinates are $q = [x_c, \theta]^T$.

**Kinematics.** The cart position is simply $\boldsymbol{p}_c = [x_c, 0]^T$. The pendulum's center of mass, located at distance $l_s$ from the pivot along the rod, has position:

$$\boldsymbol{p}_p = \begin{bmatrix} x_c - l_s \sin\theta \\ l_s \cos\theta \end{bmatrix}$$

Taking the time derivative yields the pendulum velocity:

$$\dot{\boldsymbol{p}}_p = \begin{bmatrix} \dot{x}_c - l_s \dot{\theta} \cos\theta \\ -l_s \dot{\theta} \sin\theta \end{bmatrix}$$

**Kinetic energy.** Three contributions — translational cart, translational pendulum, rotational pendulum:

$$T = \frac{1}{2} m_c \dot{x}_c^2 + \frac{1}{2} m_p \left( \dot{p}_{p,x}^2 + \dot{p}_{p,y}^2 \right) + \frac{1}{2} I_{zz} \dot{\theta}^2$$

Expanding the pendulum translational term:

$$\dot{p}_{p,x}^2 + \dot{p}_{p,y}^2 = \dot{x}_c^2 - 2 l_s \dot{x}_c \dot{\theta} \cos\theta + l_s^2 \dot{\theta}^2$$

So the total kinetic energy becomes:

$$T = \frac{1}{2}(m_c + m_p)\dot{x}_c^2 - m_p l_s \dot{x}_c \dot{\theta} \cos\theta + \frac{1}{2}(I_{zz} + m_p l_s^2)\dot{\theta}^2$$

**Potential energy.** Only the pendulum contributes (the cart moves horizontally):

$$V = m_p g l_s \cos\theta$$

**Lagrangian:** $L = T - V$.

**Rayleigh dissipation function** (models friction):

$$\mathcal{R} = \frac{1}{2} k_c \dot{x}_c^2 + \frac{1}{2} k_p \dot{\theta}^2$$

**Euler-Lagrange equations** with dissipation and external force $F$ on the cart:

$$\frac{d}{dt}\frac{\partial L}{\partial \dot{q}_i} - \frac{\partial L}{\partial q_i} + \frac{\partial \mathcal{R}}{\partial \dot{q}_i} = Q_i$$

where $Q_1 = F$ (force on cart) and $Q_2 = 0$ (no external torque on pendulum).

### 2.4 Equations of Motion (Nonlinear)

After evaluating the Euler-Lagrange equations (done symbolically in `modelSymbolic.ipynb`), the two coupled second-order ODEs are:

**Cart equation ($q_1 = x_c$):**

$$(m_c + m_p)\ddot{x}_c - m_p l_s \ddot{\theta}\cos\theta + m_p l_s \dot{\theta}^2 \sin\theta + k_c \dot{x}_c = F$$

**Pendulum equation ($q_2 = \theta$):**

$$(I_{zz} + m_p l_s^2)\ddot{\theta} - m_p l_s \ddot{x}_c \cos\theta - m_p g l_s \sin\theta + k_p \dot{\theta} = 0$$

These are **nonlinear** due to the $\sin\theta$, $\cos\theta$, and $\dot{\theta}^2$ terms. The simulation code in `pendulum.py` solves these numerically using `scipy.integrate.solve_ivp` with an RK45 integrator.

### 2.5 System Parameters

| Parameter | Symbol | Value | Unit |
|---|---|---|---|
| Cart mass | $m_c$ | 4.0 | kg |
| Pendulum mass | $m_p$ | 0.36 | kg |
| Pendulum length (to CoM) | $l_s$ | 0.451 | m |
| Gravitational acceleration | $g$ | 9.81 | m/s² |
| Pendulum moment of inertia | $I_{zz}$ | 0.08433 | kg·m² |
| Cart friction coefficient | $k_c$ | 0.1 | N·s/m |
| Pendulum friction coefficient | $k_p$ | 0.01 | N·m·s/rad |
| Maximum actuator force | $F_{\max}$ | 30 | N |
| Cart rail limits | $x_c$ | ±2 | m |

## 3. Linearization Around the Upright Equilibrium

### 3.1 Why Linearize?

PID controllers are designed for linear systems. The inverted pendulum is nonlinear, but near the upright position ($\theta \approx 0$), the nonlinear equations simplify dramatically. This is the **small-angle approximation**.

### 3.2 Small-Angle Approximation

For small $\theta$:
- $\sin\theta \approx \theta$
- $\cos\theta \approx 1$
- $\dot{\theta}^2 \sin\theta \approx 0$ (product of small quantities)

Applying these to the equations of motion:

**Linearized cart equation:**

$$(m_c + m_p)\ddot{x}_c - m_p l_s \ddot{\theta} + k_c \dot{x}_c = F$$

**Linearized pendulum equation:**

$$(I_{zz} + m_p l_s^2)\ddot{\theta} - m_p l_s \ddot{x}_c - m_p g l_s \theta + k_p \dot{\theta} = 0$$

### 3.3 Physical Interpretation

The linearized pendulum equation reveals the instability mechanism. Isolate $\ddot{\theta}$:

$$\ddot{\theta} = \frac{m_p l_s \ddot{x}_c + m_p g l_s \theta - k_p \dot{\theta}}{I_{zz} + m_p l_s^2}$$

The $m_p g l_s \theta$ term is the culprit. When $\theta > 0$ (pendulum tilts left), this term produces $\ddot{\theta} > 0$ — the pendulum accelerates further away from vertical. Gravity amplifies any deviation. Without a controller, the system is **open-loop unstable**.

## 4. PID Controller Theory

### 4.1 The Feedback Loop

A closed-loop controller continuously measures the output, computes an error signal, and applies a corrective input:

$$u(t) \xrightarrow{\text{force}} \boxed{\text{Plant}} \xrightarrow{y} \text{measurement} \xrightarrow{e = r - y} \boxed{\text{Controller}} \xrightarrow{u(t)}$$

where $r$ is the reference (desired value — basically 0) and $e$ is the error.

### 4.2 PID Control Law

The PID controller computes the control input as a sum of three terms:

$$u(t) = K_p \, e(t) + K_i \int_0^t e(\tau)\, d\tau + K_d \, \frac{de(t)}{dt}$$

Each term serves a distinct purpose:

**Proportional (P):** Generates a force proportional to the current error. If the pendulum tilts by $\theta = 0.1$ rad, the P-term pushes back with force $K_p \cdot 0.1$. Larger $K_p$ means stronger correction — but too large and the system overshoots and oscillates.

**Integral (I):** Accumulates past errors over time. If a small steady-state offset persists (the pendulum hangs at $\theta = 0.01$ rad indefinitely), the integral grows and eventually eliminates it. The I-term acts slowly. Too much $K_i$ causes windup — the integral saturates the actuator.

**Derivative (D):** Reacts to the rate of change of the error. If the pendulum is falling fast ($\dot{\theta}$ large), the D-term applies a strong corrective force before the angle itself grows large. This provides damping and reduces overshoot. Noise sensitivity is the trade-off — numerical differentiation amplifies measurement noise.

### 4.3 Discrete-Time PID

The simulation runs in discrete time steps of $\Delta t = 0.01$ s. The continuous PID must be discretized, where $k$ is the current time step index, so that time $t = k \cdot \Delta t$:

$$u[k] = K_p \, e[k] + K_i \sum_{j=0}^{k} e[j]\,\Delta t + K_d \, \frac{e[k] - e[k-1]}{\Delta t}$$

The integral becomes a running sum. The derivative becomes a backward difference.

### 4.4 Parallel PID Architecture

A single PID on $\theta$ stabilizes the pendulum angle, but ignores the cart position entirely. The cart can drift toward the rail limits at $\pm 2$ m while the pendulum stays upright.

The solution: **two parallel PID controllers** whose outputs are summed into a single force command.

1. **Angle PID (fast, dominant):** Computes a stabilizing force based on the angle error $\theta_{\text{ref}} - \theta$. This is the primary controller — it keeps the pendulum upright.

2. **Cart PID (slow, corrective):** Computes an additional force based on the position error $x_{c,\text{ref}} - x_c$. This force biases the pendulum slightly, causing it to lean and drift toward the target position.

The total control signal:

$$F = F_\theta + F_{x_c} = \text{PID}_\theta(\theta_{\text{ref}} - \theta) + \text{PID}_{x_c}(x_{c,\text{ref}} - x_c)$$

The force is then clipped to the actuator limits $\pm 30$ N.

This is structurally equivalent to a **state-feedback controller**: the combined PID terms approximate a linear combination of the full state vector $[x_c,\, \dot{x}_c,\, \theta,\, \dot{\theta}]^T$, similar to what an LQR design would produce. The P-terms act on position/angle, the D-terms approximate velocity feedback, and the I-terms eliminate steady-state error.

Unlike a cascade architecture (where the outer loop feeds a setpoint into the inner loop), the parallel structure avoids **derivative kick** — a problem where changes in the inner loop's setpoint cause violent force spikes through the derivative term.

## 5. Implementation

In [ ]:
class PIDController:
    """Discrete-time PID controller with anti-windup clamp."""

    def __init__(
        self,
        Kp: float,
        Ki: float,
        Kd: float,
        dt: float,
        output_limits: tuple[float, float] = (-np.inf, np.inf),
    ) -> None:
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.dt = dt
        self.output_limits = output_limits

        self.integral: float = 0.0
        self.prev_error: float = 0.0
        self.first_call: bool = True

    def compute(self, error: float) -> float:
        P = self.Kp * error

        # Integral with anti-windup: only accumulate if output is not saturated
        self.integral += error * self.dt
        I = self.Ki * self.integral

        # Derivative (skip on first call to avoid spike)
        if self.first_call:
            D = 0.0
            self.first_call = False
        else:
            D = self.Kd * (error - self.prev_error) / self.dt

        self.prev_error = error

        output = P + I + D

        lo, hi = self.output_limits
        if output > hi:
            # Anti-windup: undo the integral step that caused saturation
            self.integral -= error * self.dt
            output = hi
        elif output < lo:
            self.integral -= error * self.dt
            output = lo

        return output

    def reset(self) -> None:
        self.integral = 0.0
        self.prev_error = 0.0
        self.first_call = True


The `anti-windup` mechanism deserves explanation. When the actuator saturates (output hits ±30 N), the integral term keeps growing even though the system can't respond. Once the error reverses, the bloated integral causes massive overshoot. The fix: when the output is clamped, undo the last integral accumulation. The integral stays frozen at the saturation boundary until the output comes back within range.

## 6. Simulation — Step by Step

### 6.1 Baseline: No Controller (Open-Loop)

Before applying any control, observe the natural behavior. The pendulum starts at $\theta_0 = 0.1$ rad (~5.7°) with zero velocity. No force is applied ($u = 0$).

In [ ]:
# --- 1. Set the Simulation Parameters --- #
dt       = 0.01
timeSpan = 5                       # 5 seconds is enough to watch it fall
nSteps   = int(timeSpan / dt)

# --- 2. Set the Initial State of the Pendulum --- #
x0 = [0, 0, 0.1, 0]                # [cartPos, cartVel, pendAngle, pendAngleVel]

# --- 3. Create Pendulum Instance --- #
pendulum = InvertedPendulum(x0, dt=dt)

# --- 4. Create Storage Arrays --- #
y_ol = np.zeros((2, nSteps))       # output: row 0 = x_c, row 1 = θ
y_ol[:, 0] = [x0[0], x0[2]]        # store initial cart position and angle
t_ol = np.zeros(nSteps)            # time vector

# --- 5. Simulate the System --- #
for i in range(1, nSteps):
    u = 0.0                        # no control input (open-loop)
    y_tmp, _ = pendulum.step(u)    # step returns [x_c, θ], discard realized force
    y_ol[:, i] = y_tmp             # store output at timestep i
    t_ol[i] = i * dt               # record time

# --- 6. Plot Results --- #
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(t_ol, y_ol[1], color='tab:red')
ax[0].set_title("Pendulum Angle (Open-Loop)")
ax[0].set_xlabel("Time [s]")
ax[0].set_ylabel("$\\theta$ [rad]")
ax[0].axhline(0, color='k', linestyle='--', alpha=0.3)
ax[0].grid()

ax[1].plot(t_ol, y_ol[0], color='tab:blue')
ax[1].set_title("Cart Position (Open-Loop)")
ax[1].set_xlabel("Time [s]")
ax[1].set_ylabel("$x_c$ [m]")
ax[1].axhline(2, color='r', linestyle='--', alpha=0.5, label='Rail limit')
ax[1].axhline(-2, color='r', linestyle='--', alpha=0.5)
ax[1].grid()
ax[1].legend()

plt.tight_layout()
plt.show()

print(f"After {timeSpan}s: θ = {y_ol[1, -1]:.2f} rad, xc = {y_ol[0, -1]:.2f} m")


The pendulum falls within seconds. The 0.1 rad initial tilt is enough — gravity does the rest. This confirms the system is open-loop unstable and a controller is mandatory.

### 6.2 P-Only Controller

Start with just the proportional term. The control law:

$$F = K_p \cdot (0 - \theta)$$

Only $K_p$ acts — no integral, no derivative. This reveals the proportional gain's effect in isolation.

In [ ]:
# --- P-only sweep: test different Kp values --- #
dt = 0.01
timeSpan = 10
nSteps = int(timeSpan / dt)
x0 = [0, 0, 0.1, 0]

Kp_values = [10, 50, 100, 200]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for idx, Kp in enumerate(Kp_values):
    pendulum = InvertedPendulum(x0, dt=dt)

    y_arr = np.zeros((2, nSteps))
    t_arr = np.zeros(nSteps)
    y_arr[:, 0] = [x0[0], x0[2]]

    for i in range(1, nSteps):
        theta_current = y_arr[1, i-1]
        error = 0.0 - theta_current
        u = Kp * error

        y_tmp, uR = pendulum.step(u)
        y_arr[:, i] = y_tmp
        t_arr[i] = i * dt

    ax = axes[idx]
    ax.plot(t_arr, y_arr[1], color='tab:orange', label='$\\theta$')
    ax.axhline(0, color='k', linestyle='--', alpha=0.3)
    ax.set_title(f"$K_p$ = {Kp}")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("$\\theta$ [rad]")
    ax.set_ylim([-0.5, 0.5])
    ax.grid()
    ax.legend()

plt.suptitle("P-Only Controller — Effect of $K_p$", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

The P-only controller shows the classic trade-off: too little gain and the pendulum falls; enough gain and it oscillates around zero. The oscillations persist because there is no damping mechanism — the D-term is missing.

### 6.3 PD Controller (Adding Damping)

Adding the derivative term provides damping. The D-term opposes the rate of change: when the pendulum swings toward vertical, the derivative brakes it, preventing overshoot.

$$F = K_p(0 - \theta) + K_d \frac{d}{dt}(0 - \theta) \approx K_p(-\theta) + K_d \frac{(-\theta[k]) - (-\theta[k-1])}{\Delta t}$$

In [ ]:
# --- PD controller sweep: fix Kp, vary Kd --- #
dt = 0.01
timeSpan = 10
nSteps = int(timeSpan / dt)
x0 = [0, 0, 0.1, 0]

Kp_fixed = 100
Kd_values = [1, 5, 10, 20]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for idx, Kd in enumerate(Kd_values):
    pendulum = InvertedPendulum(x0, dt=dt)
    pid_theta = PIDController(Kp_fixed, 0, Kd, dt, output_limits=(-30, 30))

    y_arr = np.zeros((2, nSteps))
    t_arr = np.zeros(nSteps)
    y_arr[:, 0] = [x0[0], x0[2]]

    for i in range(1, nSteps):
        theta_current = y_arr[1, i-1]
        error = 0.0 - theta_current
        u = pid_theta.compute(error)

        y_tmp, uR = pendulum.step(u)
        y_arr[:, i] = y_tmp
        t_arr[i] = i * dt

    ax = axes[idx]
    ax.plot(t_arr, y_arr[1], color='tab:orange', label='$\\theta$')
    ax.plot(t_arr, y_arr[0], color='tab:blue', label='$x_c$', alpha=0.7)
    ax.axhline(0, color='k', linestyle='--', alpha=0.3)
    ax.set_title(f"$K_p={Kp_fixed}$, $K_d={Kd}$")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Value")
    ax.set_ylim([-1, 1])
    ax.grid()
    ax.legend(loc='upper right')

plt.suptitle("PD Controller — Effect of $K_d$ (Damping)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

The D-term eliminates oscillations. The pendulum converges to $\theta = 0$ without bouncing. But look at the cart position (blue line) — it drifts. The angle controller has no concept of where the cart is. Time to add a second PID for cart position.

### 6.4 Full Parallel PID Controller

Two PID controllers running in parallel, their outputs summed:

**Angle PID** — Pendulum stabilization (dominant):
$$F_\theta = \text{PID}_\theta(0 - \theta)$$

**Cart PID** — Position correction (additive):
$$F_{x_c} = \text{PID}_{x_c}(0 - x_c)$$

**Total force:**
$$F = \text{clip}(F_\theta + F_{x_c},\; -30,\; 30)$$

The angle PID has **positive** gains — when the pendulum tilts left ($\theta > 0$), it pushes the cart left (under the pendulum) to catch it. The cart PID has **negative** gains: when the cart is too far right, it adds a small rightward force. This tilts the pendulum left, and the much stronger angle PID reacts by pushing the cart left — back toward the origin. The cart PID works *through* the pendulum dynamics, not against them.

Gain selection rationale:
- Angle PID needs **fast** response: $K_p = 150$, $K_d = 15$
- Cart PID needs **gentle** correction: $|K_p| = 5$, $|K_d| = 5$ — strong enough to prevent drift, weak enough not to destabilize the angle control
- No individual output limits on either PID — only the total force is clipped to the actuator range

In [ ]:
dt = 0.01
timeSpan = 30        # 30 seconds to demonstrate long-term stability
nSteps = int(timeSpan / dt)

# Initial condition: moderate tilt, cart at origin
x0 = [0, 0, 0.3, 0]
pendulum = InvertedPendulum(x0, dt=dt)

# --- Angle PID (stabilize pendulum) --- #
pid_theta = PIDController(
    Kp=150, Ki=0.5, Kd=15,
    dt=dt,
    output_limits=(-np.inf, np.inf)
)

# --- Cart PID (return cart to origin) --- #
pid_cart = PIDController(
    Kp=-5, Ki=-0.05, Kd=-5,
    dt=dt,
    output_limits=(-np.inf, np.inf)
)

# --- Target --- #
xc_target = 0.0    # keep cart at origin
theta_target = 0.0 # keep pendulum upright

# --- Storage --- #
y_cl    = np.zeros((2, nSteps))
u_log   = np.zeros((2, nSteps))  # [commanded, realized]
t_cl    = np.zeros(nSteps)

y_cl[:, 0] = [x0[0], x0[2]]

# --- Simulation loop --- #
for i in range(1, nSteps):
    xc_current    = y_cl[0, i-1]
    theta_current = y_cl[1, i-1]

    # Parallel PID: both produce force, sum is clipped to actuator limits
    F_theta = pid_theta.compute(theta_target - theta_current)
    F_cart  = pid_cart.compute(xc_target - xc_current)
    F = np.clip(F_theta + F_cart, -30, 30)

    # Step the plant
    y_tmp, uRealized = pendulum.step(F)

    # Store
    y_cl[:, i]  = y_tmp
    u_log[0, i] = F
    u_log[1, i] = uRealized
    t_cl[i]     = i * dt

print(f"Final state: θ = {y_cl[1, -1]:.6f} rad, xc = {y_cl[0, -1]:.4f} m")
print(f"Max |θ| = {np.max(np.abs(y_cl[1])):.4f} rad")
print(f"Max |xc| = {np.max(np.abs(y_cl[0])):.4f} m (limit: 2.0 m)")

In [ ]:
# --- Comprehensive result plot --- #
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# --- Subplot 1: Pendulum Angle --- #
axes[0].plot(t_cl, y_cl[1], color='tab:orange', label='$\\theta$ (actual)', linewidth=1.2)
axes[0].axhline(0, color='k', linestyle=':', alpha=0.3)
axes[0].set_ylabel('$\\theta$ [rad]')
axes[0].set_title('Pendulum Angle')
axes[0].legend(loc='upper right')
axes[0].grid(alpha=0.3)

# --- Subplot 2: Cart Position --- #
axes[1].plot(t_cl, y_cl[0], color='tab:blue', label='$x_c$', linewidth=1.2)
axes[1].axhline(2, color='r', linestyle='--', alpha=0.5, label='Rail limit')
axes[1].axhline(-2, color='r', linestyle='--', alpha=0.5)
axes[1].axhline(0, color='k', linestyle=':', alpha=0.3)
axes[1].set_ylabel('$x_c$ [m]')
axes[1].set_title('Cart Position')
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

# --- Subplot 3: Control Input --- #
axes[2].plot(t_cl, u_log[0], color='tab:red', label='$F$ (commanded)', alpha=0.7, linewidth=1.0)
axes[2].plot(t_cl, u_log[1], color='tab:green', label='$F_{realized}$', linewidth=1.2, linestyle='--')
axes[2].axhline(30, color='gray', linestyle='--', alpha=0.4, label='Actuator limit')
axes[2].axhline(-30, color='gray', linestyle='--', alpha=0.4)
axes[2].set_ylabel('$F$ [N]')
axes[2].set_xlabel('Time [s]')
axes[2].set_title('Control Force')
axes[2].legend(loc='upper right')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Animation

Visual confirmation of the stabilization. The pendulum starts tilted and returns to vertical while the cart stays within the safe zone.

In [ ]:
# --- Animate the nominal case (30s parallel PID result) --- #
# Use shorter time for faster rendering
anim_time = 15  # seconds
anim_steps = int(anim_time / dt)

showAnimation(t_cl[:anim_steps], y_cl[0, :anim_steps], y_cl[1, :anim_steps])

## 8. Robustness Test

The controller should handle different initial conditions — not just $\theta_0 = 0.3$ rad from rest. Test with cart offsets, initial velocities, and combined perturbations to verify robustness.

In [ ]:
# --- Robustness: multiple initial conditions --- #
test_cases = [
    {"label": "θ₀ = 0.1 rad (small tilt)",          "x0": [0, 0, 0.1, 0]},
    {"label": "xc₀ = 1.5 m, θ₀ = 0.2 rad",         "x0": [1.5, 0, 0.2, 0]},
    {"label": "ẋ₀ = 2.0 m/s, θ₀ = 0.2 rad",        "x0": [0, 2.0, 0.2, 0]},
    {"label": "xc₀ = -1 m, ẋ₀ = 0.5, θ₀ = 0.3",   "x0": [-1.0, 0.5, 0.3, 0]},
]

dt = 0.01
timeSpan = 20
nSteps = int(timeSpan / dt)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for idx, tc in enumerate(test_cases):
    x0_test = tc["x0"]
    pendulum = InvertedPendulum(x0_test, dt=dt)

    pid_t = PIDController(150, 0.5, 15, dt, output_limits=(-np.inf, np.inf))
    pid_c = PIDController(-5, -0.05, -5, dt, output_limits=(-np.inf, np.inf))

    y_arr = np.zeros((2, nSteps))
    t_arr = np.zeros(nSteps)
    y_arr[:, 0] = [x0_test[0], x0_test[2]]

    for i in range(1, nSteps):
        F_theta = pid_t.compute(0.0 - y_arr[1, i-1])
        F_cart  = pid_c.compute(0.0 - y_arr[0, i-1])
        F = np.clip(F_theta + F_cart, -30, 30)

        y_tmp, _ = pendulum.step(F)
        y_arr[:, i] = y_tmp
        t_arr[i] = i * dt

    ax = axes[idx]
    ax.plot(t_arr, y_arr[1], color='tab:orange', label='$\\theta$ [rad]')
    ax.plot(t_arr, y_arr[0], color='tab:blue', label='$x_c$ [m]')
    ax.axhline(0, color='k', linestyle=':', alpha=0.3)
    ax.axhline(2, color='r', linestyle='--', alpha=0.3)
    ax.axhline(-2, color='r', linestyle='--', alpha=0.3)
    ax.set_title(tc["label"])
    ax.set_xlabel("Time [s]")
    ax.grid(alpha=0.3)
    ax.legend(loc='upper right')
    ax.set_ylim([-2.5, 2.5])

plt.suptitle("Robustness Test — Various Initial Conditions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Animate all robustness test cases --- #
from IPython.display import display, HTML as RawHTML

anim_cases = [
    {"label": "θ₀ = 0.1 rad (small tilt)",          "x0": [0, 0, 0.1, 0]},
    {"label": "xc₀ = 1.5 m, θ₀ = 0.2 rad",         "x0": [1.5, 0, 0.2, 0]},
    {"label": "ẋ₀ = 2.0 m/s, θ₀ = 0.2 rad",        "x0": [0, 2.0, 0.2, 0]},
    {"label": "xc₀ = −1 m, ẋ₀ = 0.5, θ₀ = 0.3",   "x0": [-1.0, 0.5, 0.3, 0]},
]

dt = 0.01
timeSpan = 15
nSteps = int(timeSpan / dt)

for tc in anim_cases:
    x0_a = tc["x0"]
    pend = InvertedPendulum(x0_a, dt=dt)
    pid_t = PIDController(150, 0.5, 15, dt, output_limits=(-np.inf, np.inf))
    pid_c = PIDController(-5, -0.05, -5, dt, output_limits=(-np.inf, np.inf))

    y_a = np.zeros((2, nSteps))
    t_a = np.zeros(nSteps)
    y_a[:, 0] = [x0_a[0], x0_a[2]]

    for i in range(1, nSteps):
        Ft = pid_t.compute(0.0 - y_a[1, i-1])
        Fc = pid_c.compute(0.0 - y_a[0, i-1])
        F = np.clip(Ft + Fc, -30, 30)
        yt, _ = pend.step(F)
        y_a[:, i] = yt
        t_a[i] = i * dt

    display(RawHTML(f"<h4>{tc['label']}</h4>"))
    display(showAnimation(t_a, y_a[0], y_a[1]))

## 9. Gain Tuning Methodology

### 9.1 How These Gains Were Found

The tuning followed a manual process grounded in the physics:

**Step 1 — Angle PID $K_p$.** Start with $K_d = 0$, $K_i = 0$. Increase $K_p$ until the pendulum oscillates around $\theta = 0$ without falling. The gravity torque near vertical is approximately $m_p g l_s \theta \approx 1.59\,\theta$ N·m. The controller must overcome this through the cart dynamics, so $K_p$ needs to be well above 50. A value of **150** worked.

**Step 2 — Angle PID $K_d$ (damping).** With $K_p = 150$, the pendulum oscillates. Add $K_d$ to damp the oscillations. $K_d = 15$ (roughly $K_p / 10$) provides critical damping — the angle converges in about 2 seconds with minimal overshoot.

**Step 3 — Cart PID $K_p$ (position correction).** With the pendulum stabilized, the cart drifts freely. Add a cart PID with **negative** gains. The sign is intentional: when the cart is displaced right ($x_c > 0$), the error is $0 - x_c < 0$, and $K_p = -5$ produces a *positive* (rightward) force. This does not push the cart back directly — instead, it tilts the pendulum slightly left ($\Delta\theta > 0$). The angle PID, which is 30× stronger, reacts by pushing the cart left to catch the tilt. The net force is leftward, driving the cart back toward the origin. $K_p^{\text{cart}} = -5$ provides effective position control without destabilizing the angle loop.

**Step 4 — Cart PID $K_d$ (position damping).** Prevents the cart from overshooting the target position. $K_d^{\text{cart}} = -5$ damps the cart's return trajectory.

**Step 5 — Integral terms.** Small integral gains ($K_i^\theta = 0.5$, $K_i^{\text{cart}} = -0.05$) eliminate any residual steady-state error. These are small enough to avoid windup.

### 9.2 Why Parallel Instead of Cascade?

A cascade design (outer loop outputs $\theta_{\text{ref}}$, inner loop tracks it) is a standard technique in industrial control. However, it requires two conditions that our implementation does not meet:

1. **Time-scale separation.** The inner loop must run at a significantly higher frequency than the outer loop (e.g., 1 kHz vs. 100 Hz). In our simulation both loops execute at the same rate ($\Delta t = 0.01$ s), so the outer loop updates $\theta_{\text{ref}}$ every step — and the inner loop sees these changes as abrupt jumps.

2. **Derivative-on-measurement.** Industrial PID controllers typically compute the D-term from the process variable, not from the error: $D = -K_d \cdot \Delta\theta / \Delta t$ instead of $D = K_d \cdot \Delta(\theta_{\text{ref}} - \theta) / \Delta t$. This eliminates sensitivity to setpoint changes. Our `PIDController` class uses derivative-on-error.

Without these two safeguards, every change in $\theta_{\text{ref}}$ is amplified by the inner loop's D-term ($K_d / \Delta t$), causing violent force oscillations (**derivative kick**). In testing, the force oscillated between $\pm 30$ N on every step — the system was unstable.

The parallel structure sidesteps both issues: each PID's setpoint is constant ($0$), so the derivative term only reacts to smooth changes in the process variables ($\theta$, $x_c$). No time-scale separation is needed because the two controllers do not feed into each other.


## 10. Performance Summary

In [ ]:
# --- Compute metrics --- #
threshold = 0.01
settled = np.abs(y_cl[1]) < threshold
settling_idx = None
for i in range(len(settled)):
    if np.all(settled[i:]):
        settling_idx = i
        break

settling_time = t_cl[settling_idx] if settling_idx is not None else float('inf')
overshoot = abs(np.min(y_cl[1]))
max_xc = np.max(np.abs(y_cl[0]))
max_force = np.max(np.abs(u_log[1]))
ss_theta = np.mean(np.abs(y_cl[1, -500:]))
ss_xc = np.mean(np.abs(y_cl[0, -500:]))
within_limits = max_xc < 2.0

# --- Render as matplotlib table --- #
rows = [
    ["Settling time (|θ| < 0.01 rad)",  f"{settling_time:.2f} s"],
    ["Peak overshoot (θ)",              f"{overshoot:.4f} rad  ({np.degrees(overshoot):.1f}°)"],
    ["Max cart displacement",            f"{max_xc:.4f} m"],
    ["Max actuator force",              f"{max_force:.1f} / 30.0 N"],
    ["Steady-state |θ| (last 5 s)",     f"{ss_theta:.6f} rad"],
    ["Steady-state |xc| (last 5 s)",    f"{ss_xc:.6f} m"],
    ["Cart within ±2 m",                "Yes" if within_limits else "No"],
]

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off')

table = ax.table(
    cellText=rows,
    colLabels=['Metric', 'Value'],
    cellLoc='left',
    colColours=['#4a90d9', '#4a90d9'],
    loc='center',
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.0, 1.6)

# Style header
for j in range(2):
    table[0, j].set_text_props(color='white', fontweight='bold')

# Alternate row colors
for i in range(1, len(rows) + 1):
    color = '#f0f4fa' if i % 2 == 0 else 'white'
    for j in range(2):
        table[i, j].set_facecolor(color)
        table[i, j].set_edgecolor('#cccccc')

plt.title('Controller Performance', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## 11. Conclusion

### What Works

The parallel PID controller stabilizes the inverted pendulum from a 0.3 rad initial perturbation within a few seconds. The cart stays well within the ±2 m rail limits. The control force remains within the ±30 N actuator saturation. The integral terms eliminate steady-state drift in both angle and position.

The parallel architecture avoids the derivative kick problem that plagues cascade designs: since both PID setpoints are constant (zero), the D-terms only respond to smooth process variable changes.

### Limitations

PID operates on measured outputs only ($x_c$, $\theta$) — the velocities $\dot{x}_c$ and $\dot{\theta}$ are approximated through the D-term's finite differences. A state observer (Luenberger, Kalman filter) would provide cleaner velocity estimates.

The linearization assumption ($\sin\theta \approx \theta$) restricts the controller to small angles. For $\theta > 0.3$ rad (~17°), the approximation breaks down and performance degrades. Task 3 (swing-up from $\theta = \pi$) requires a fundamentally different approach — energy-based swing-up control, nonlinear MPC or RL.

### Connection to Task 2

**Task 2** (move cart to new position while stabilized) requires only a change in the cart PID reference: set $x_{c,\text{ref}} \neq 0$. The parallel structure supports this directly — the cart PID will bias the force to drive the cart toward the new target.